# PROJECT 1 : PDF CHATBOT USING RAG + GEMINI

Author : Kunal Kirtak

Tech Stack
- Gemini 2.5 Flash
- Sentence Transformers
- FAISS
- PyMuPDF
- LangChain
- Streamlit


## Part 1 : Project Foundation


In [58]:
print("="*60)
print("PDF CHATBOT USING RAG")
print("="*60)

PDF CHATBOT USING RAG


In [59]:
from pathlib import Path

PROJECT_NAME = "01-PDF-Chatbot"

ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME

folders = [
    ROOT,
    ROOT / "app",
    ROOT / "config",
    ROOT / "core",
    ROOT / "data",
    ROOT / "data" / "raw",
    ROOT / "data" / "processed",
    ROOT / "embeddings",
    ROOT / "vector_store",
    ROOT / "logs",
    ROOT / "assets",
    ROOT / "screenshots",
    ROOT / "notebooks",
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Project created successfully!")

print(ROOT)

Project created successfully!
/content/drive/MyDrive/01-PDF-Chatbot


In [60]:
!pip install -q \
google-generativeai==0.8.5 \
sentence-transformers==5.1.0 \
faiss-cpu==1.12.0 \
PyMuPDF==1.26.4 \
streamlit==1.49.1 \
python-dotenv==1.1.1 \
tqdm==4.67.1 \
langchain==0.3.27 \
langchain-community==0.3.29

In [61]:
import fitz
import faiss
import streamlit
import google.generativeai as genai

print("Everything imported successfully!")

Everything imported successfully!


In [62]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter Gemini API Key: ")

Enter Gemini API Key: ··········


In [63]:
!pip install -q google-genai

In [64]:
from google import genai
import os

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gemini-robotics-er-2-preview
models/gemini-2.5-computer-use-preview-10-2025
models/an

In [65]:
from google import genai
import os

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Say hello in one sentence."
)

print(response.text)

Hello and welcome, I hope you are having a wonderful day!


In [66]:
requirements = """
google-generativeai
langchain
langchain-community
langchain-text-splitters
sentence-transformers
faiss-cpu
pymupdf
python-dotenv
streamlit
tqdm
"""

(ROOT / "requirements.txt").write_text(requirements.strip())

print("requirements.txt created")

requirements.txt created


In [67]:
gitignore = """
__pycache__/
*.pyc
.env
.ipynb_checkpoints/
vector_store/
logs/
"""

(ROOT / ".gitignore").write_text(gitignore.strip())

print(".gitignore created")

.gitignore created


In [68]:
logger_code = '''
import logging
from pathlib import Path

LOG_DIR = Path(__file__).resolve().parent.parent / "logs"
LOG_DIR.mkdir(exist_ok=True)

logging.basicConfig(
    filename=LOG_DIR / "app.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("PDFChatbot")
'''

(ROOT / "core" / "logger.py").write_text(logger_code)

print("logger.py created")

logger.py created


In [69]:
for path in ROOT.rglob("*"):
    print(path.relative_to(ROOT))

cloudflared
app
config
core
data
embeddings
vector_store
logs
assets
screenshots
notebooks
.ipynb_checkpoints
chat.py
.gitignore
ingest.py
chunk_documents.py
build_vector_store.py
search_demo.py
app.py
Dockerfile
LICENSE
README.md
requirements.txt
config/.ipynb_checkpoints
config/config.py
core/.ipynb_checkpoints
core/__pycache__
core/retriever.py
core/prompt_builder.py
core/logger.py
core/utils.py
core/chunker.py
core/embedding.py
core/vector_store.py
core/prompt.py
core/gemini_client.py
core/rag_pipeline.py
core/pdf_loader.py
data/raw
data/processed
data/chunks
vector_store/faiss_index.bin
vector_store/metadata.json
core/__pycache__/retriever.cpython-312.pyc
core/__pycache__/prompt_builder.cpython-312.pyc
core/__pycache__/pdf_loader.cpython-312.pyc
core/__pycache__/utils.cpython-312.pyc
core/__pycache__/chunker.cpython-312.pyc
core/__pycache__/embedding.cpython-312.pyc
core/__pycache__/vector_store.cpython-312.pyc
core/__pycache__/rag_pipeline.cpython-312.pyc
core/__pycache__/prompt.

## Part 2 — PDF Ingestion Pipeline

In [70]:
from pathlib import Path
import json
import fitz
import re
from tqdm import tqdm

In [71]:
utils_code = '''
import re

def clean_text(text: str) -> str:
    """
    Clean extracted PDF text.
    """

    text = re.sub(r'\\n+', '\\n', text)

    text = re.sub(r'\\s+', ' ', text)

    text = text.strip()

    return text


def word_count(text):

    return len(text.split())


def character_count(text):

    return len(text)
'''

(ROOT/"core"/"utils.py").write_text(utils_code)

print("utils.py created")

utils.py created


In [72]:
loader_code = '''
import fitz
from pathlib import Path

from core.utils import clean_text

class PDFLoader:

    def __init__(self, pdf_path):

        self.pdf_path = Path(pdf_path)

    def extract_text(self):

        document = fitz.open(self.pdf_path)

        pages=[]

        for page in document:

            text = page.get_text()

            pages.append(clean_text(text))

        document.close()

        return pages

    def page_count(self):

        document=fitz.open(self.pdf_path)

        total=len(document)

        document.close()

        return total
'''

(ROOT/"core"/"pdf_loader.py").write_text(loader_code)

print("pdf_loader.py created")

pdf_loader.py created


In [73]:
ingest_code = '''
from pathlib import Path
import json

from tqdm import tqdm

from core.pdf_loader import PDFLoader
from core.utils import word_count, character_count

RAW_DIR = Path("data/raw")

PROCESSED_DIR = Path("data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def process_pdf(pdf_path):

    loader = PDFLoader(pdf_path)

    pages = loader.extract_text()

    page_data=[]

    for page_number,text in enumerate(pages,start=1):

        page_data.append({

            "page":page_number,

            "text":text,

            "words":word_count(text),

            "characters":character_count(text)

        })

    document={

        "filename":Path(pdf_path).name,

        "pages":loader.page_count(),

        "content":page_data

    }

    output=PROCESSED_DIR/(Path(pdf_path).stem+".json")

    with open(output,"w",encoding="utf-8") as f:

        json.dump(document,f,indent=4,ensure_ascii=False)

    print(f"Saved -> {output}")


def main():

    pdfs=list(RAW_DIR.glob("*.pdf"))

    if not pdfs:

        print("No PDFs found.")

        return

    for pdf in tqdm(pdfs):

        process_pdf(pdf)


if __name__=="__main__":

    main()
'''

(ROOT/"ingest.py").write_text(ingest_code)

print("ingest.py created")

ingest.py created


In [74]:
import shutil
from pathlib import Path

SOURCE = Path("/content")

for pdf in SOURCE.glob("*.pdf"):
    shutil.copy(pdf, ROOT/"data"/"raw")

print("PDFs copied.")

PDFs copied.


In [75]:
%cd "{ROOT}"

/content/drive/MyDrive/01-PDF-Chatbot


In [76]:
!python ingest.py

  0% 0/1 [00:00<?, ?it/s]Saved -> data/processed/RAG.json
100% 1/1 [00:00<00:00,  3.01it/s]


In [77]:
import json

files=list((ROOT/"data"/"processed").glob("*.json"))

print(files)

[PosixPath('/content/drive/MyDrive/01-PDF-Chatbot/data/processed/RAG.json')]


In [78]:
with open(files[0],encoding="utf-8") as f:

    data=json.load(f)

print(data["filename"])

print(data["pages"])

print(data["content"][0]["text"][:500])

RAG.pdf
21
1 Retrieval-Augmented Generation for Large Language Models: A Survey Yunfan Gaoa, Yun Xiongb, Xinyu Gaob, Kangxiang Jiab, Jinliu Panb, Yuxi Bic, Yi Daia, Jiawei Suna, Meng Wangc, and Haofen Wang a,c aShanghai Research Institute for Intelligent Autonomous Systems, Tongji University bShanghai Key Laboratory of Data Science, School of Computer Science, Fudan University cCollege of Design and Innovation, Tongji University Abstract—Large Language Models (LLMs) showcase impres- sive capabilities but e


In [79]:
import json

processed=list((ROOT/"data"/"processed").glob("*.json"))

total_pages=0
total_words=0

for file in processed:

    with open(file,encoding="utf-8") as f:

        d=json.load(f)

    total_pages+=d["pages"]

    for page in d["content"]:

        total_words+=page["words"]

print("="*50)
print("Processed PDFs :",len(processed))
print("Total Pages    :",total_pages)
print("Total Words    :",total_words)
print("="*50)

Processed PDFs : 1
Total Pages    : 21
Total Words    : 15618


## Part 3 — Intelligent Chunking

In [80]:
(ROOT/"data"/"chunks").mkdir(parents=True, exist_ok=True)

print("Chunks directory created.")

Chunks directory created.


In [81]:
chunk_config = """

# ==========================
# Chunk Settings
# ==========================

CHUNK_SIZE = 500

CHUNK_OVERLAP = 100

CHUNK_STRATEGY = "recursive"

"""

In [82]:
config_path = ROOT/"config"/"config.py"

with open(config_path, "a") as f:
    f.write(chunk_config)

print("config.py updated.")

config.py updated.


In [83]:
chunker_code = '''
from langchain_text_splitters import RecursiveCharacterTextSplitter

class Chunker:

    def __init__(self,
                 chunk_size=500,
                 chunk_overlap=100):

        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def fixed_chunks(self, text):

        chunks=[]

        start=0

        while start < len(text):

            end=start+self.chunk_size

            chunks.append(text[start:end])

            start=end

        return chunks

    def sliding_chunks(self,text):

        chunks=[]

        start=0

        while start < len(text):

            end=start+self.chunk_size

            chunks.append(text[start:end])

            start += self.chunk_size-self.chunk_overlap

        return chunks

    def recursive_chunks(self,text):

        splitter=RecursiveCharacterTextSplitter(

            chunk_size=self.chunk_size,

            chunk_overlap=self.chunk_overlap

        )

        return splitter.split_text(text)
'''

(ROOT/"core"/"chunker.py").write_text(chunker_code)

print("chunker.py created.")

chunker.py created.


In [84]:
chunk_code = '''
from pathlib import Path
import json

from tqdm import tqdm

from core.chunker import Chunker

PROCESSED = Path("data/processed")
OUTPUT = Path("data/chunks")

OUTPUT.mkdir(exist_ok=True)

chunker = Chunker()

strategy="recursive"


for file in tqdm(PROCESSED.glob("*.json")):

    with open(file,encoding="utf-8") as f:

        document=json.load(f)

    chunks=[]

    chunk_id=1

    for page in document["content"]:

        text=page["text"]

        if strategy=="fixed":

            result=chunker.fixed_chunks(text)

        elif strategy=="sliding":

            result=chunker.sliding_chunks(text)

        else:

            result=chunker.recursive_chunks(text)

        for chunk in result:

            chunks.append({

                "chunk_id":chunk_id,

                "page":page["page"],

                "text":chunk,

                "length":len(chunk)

            })

            chunk_id += 1

    output={

        "filename":document["filename"],

        "strategy":strategy,

        "total_chunks":len(chunks),

        "chunks":chunks

    }

    save_path=OUTPUT/(file.stem+"_chunks.json")

    with open(save_path,"w",encoding="utf-8") as out:

        json.dump(output,out,indent=4,ensure_ascii=False)

print("Chunking completed.")
'''

(ROOT/"chunk_documents.py").write_text(chunk_code)

print("chunk_documents.py created.")

chunk_documents.py created.


In [85]:
%cd "{ROOT}"

/content/drive/MyDrive/01-PDF-Chatbot


In [86]:
!python chunk_documents.py

1it [00:00, 13.25it/s]
Chunking completed.


In [87]:
import json

files=list((ROOT/"data"/"chunks").glob("*_chunks.json"))

print(files)

[PosixPath('/content/drive/MyDrive/01-PDF-Chatbot/data/chunks/Encrypted AI Techniques for Anomaly Detection_chunks.json'), PosixPath('/content/drive/MyDrive/01-PDF-Chatbot/data/chunks/RAG_chunks.json')]


In [88]:
with open(files[0],encoding="utf-8") as f:

    data=json.load(f)

print("Strategy :",data["strategy"])

print("Chunks :",data["total_chunks"])

print()

print(data["chunks"][0]["text"])

Strategy : recursive
Chunks : 69

International Journal of Research and Review Techniques (IJRRT), ISSN: 3006-1075 Volume 3, Issue 1, January-March, 2024, Available online at: https://ijrrt.com 76 "Encrypted AI Techniques for Anomaly Detection" M. B. Farbman Israel Institute of Technology, Israel ABSTRACT The integration of artificial intelligence (AI) with encryption techniques has emerged as a pivotal area in enhancing data security and anomaly detection capabilities. This paper explores the convergence of encrypted AI


In [89]:
from core.chunker import Chunker

sample = data["chunks"][0]["text"] * 5

chunker = Chunker()

fixed = chunker.fixed_chunks(sample)

recursive = chunker.recursive_chunks(sample)

sliding = chunker.sliding_chunks(sample)

print("="*40)

print("Fixed      :",len(fixed))

print("Recursive  :",len(recursive))

print("Sliding    :",len(sliding))

Fixed      : 5
Recursive  : 6
Sliding    : 7


## Part 4 — Embeddings & FAISS Vector Database

In [90]:
embedding_code = '''
from sentence_transformers import SentenceTransformer

class EmbeddingModel:

    def __init__(self,
                 model_name="all-MiniLM-L6-v2"):

        self.model = SentenceTransformer(model_name)

    def encode(self, texts):

        if isinstance(texts, str):
            texts = [texts]

        vectors = self.model.encode(
            texts,
            convert_to_numpy=True,
            show_progress_bar=False,
            normalize_embeddings=True
        )

        return vectors
'''

(ROOT/"core"/"embedding.py").write_text(embedding_code)

print("embedding.py created.")

embedding.py created.


In [91]:
vector_code = '''
import json
from pathlib import Path

import faiss
import numpy as np

class VectorStore:

    def __init__(self):

        self.index = None
        self.metadata = []

    def build(self, embeddings, metadata):

        dimension = embeddings.shape[1]

        self.index = faiss.IndexFlatIP(dimension)

        self.index.add(embeddings.astype("float32"))

        self.metadata = metadata

    def save(self, directory):

        directory = Path(directory)

        directory.mkdir(parents=True, exist_ok=True)

        faiss.write_index(
            self.index,
            str(directory/"faiss_index.bin")
        )

        with open(directory/"metadata.json","w",encoding="utf-8") as f:

            json.dump(
                self.metadata,
                f,
                indent=4,
                ensure_ascii=False
            )

    def load(self,directory):

        directory=Path(directory)

        self.index = faiss.read_index(
            str(directory/"faiss_index.bin")
        )

        with open(directory/"metadata.json",encoding="utf-8") as f:

            self.metadata=json.load(f)

    def search(self,query_vector,k=5):

        scores,indices=self.index.search(
            query_vector.astype("float32"),
            k
        )

        results=[]

        for score,idx in zip(scores[0],indices[0]):

            if idx==-1:
                continue

            item=self.metadata[idx]

            item["score"]=float(score)

            results.append(item)

        return results
'''

(ROOT/"core"/"vector_store.py").write_text(vector_code)

print("vector_store.py created.")

vector_store.py created.


In [92]:
build_code = '''
from pathlib import Path
import json
import numpy as np

from tqdm import tqdm

from core.embedding import EmbeddingModel
from core.vector_store import VectorStore

CHUNK_DIR = Path("data/chunks")

VECTOR_DIR = Path("vector_store")

embedder = EmbeddingModel()

texts=[]

metadata=[]

for file in tqdm(CHUNK_DIR.glob("*_chunks.json")):

    with open(file,encoding="utf-8") as f:

        document=json.load(f)

    for chunk in document["chunks"]:

        texts.append(chunk["text"])

        metadata.append({

            "filename":document["filename"],

            "page":chunk["page"],

            "chunk_id":chunk["chunk_id"],

            "text":chunk["text"]

        })

print(f"Encoding {len(texts)} chunks...")

embeddings = embedder.encode(texts)

store = VectorStore()

store.build(
    embeddings,
    metadata
)

store.save(VECTOR_DIR)

print("Vector database created.")
'''

(ROOT/"build_vector_store.py").write_text(build_code)

print("build_vector_store.py created.")

build_vector_store.py created.


In [93]:
!python build_vector_store.py

2026-08-12 16:19:20.901319: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2it [00:00, 317.21it/s]
Encoding 349 chunks...
Vector database created.


In [94]:
from pathlib import Path

vector_dir = ROOT/"vector_store"

for file in vector_dir.iterdir():
    print(file.name)

faiss_index.bin
metadata.json


In [95]:
search_code = '''
from core.embedding import EmbeddingModel
from core.vector_store import VectorStore

embedder = EmbeddingModel()

store = VectorStore()

store.load("vector_store")

question = input("Question: ")

query = embedder.encode(question)

results = store.search(query,k=5)

print()

print("="*80)

for i,result in enumerate(results,1):

    print(f"Result {i}")

    print("Score :",round(result["score"],4))

    print("File :",result["filename"])

    print("Page :",result["page"])

    print()

    print(result["text"][:500])

    print()

    print("-"*80)
'''

(ROOT/"search_demo.py").write_text(search_code)

print("search_demo.py created.")

search_demo.py created.


In [96]:
!python search_demo.py

2026-08-12 16:19:38.236688: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Question: What is RAG?

Result 1
Score : 0.7819
File : RAG.pdf
Page : 1

Our contributions are as follows: • In this survey, we present a thorough and systematic review of the state-of-the-art RAG methods, delineating its evolution through paradigms including naive RAG, arXiv:2312.10997v5 [cs.CL] 27 Mar 2024

--------------------------------------------------------------------------------
Result 2
Score : 0.734
File : RAG.pdf
Page : 2

news articles related to the user’s query. These articles, combined with the original question, form a comprehensive prompt that empowers LLMs to generate a well-informed answer. The RAG research paradigm is continuously evolving, and we ca

In [97]:
import json

metadata_file = ROOT/"vector_store"/"metadata.json"

with open(metadata_file,encoding="utf-8") as f:
    metadata = json.load(f)

print("="*50)
print("Total vectors :", len(metadata))
print("Sample chunk")
print("="*50)
print(metadata[0]["text"][:500])

Total vectors : 349
Sample chunk
International Journal of Research and Review Techniques (IJRRT), ISSN: 3006-1075 Volume 3, Issue 1, January-March, 2024, Available online at: https://ijrrt.com 76 "Encrypted AI Techniques for Anomaly Detection" M. B. Farbman Israel Institute of Technology, Israel ABSTRACT The integration of artificial intelligence (AI) with encryption techniques has emerged as a pivotal area in enhancing data security and anomaly detection capabilities. This paper explores the convergence of encrypted AI


## Part 5 — Complete RAG Pipeline (Gemini + FAISS)

In [98]:
config_update = """

# ============================
# Retrieval
# ============================

TOP_K = 5

TEMPERATURE = 0.2

MAX_OUTPUT_TOKENS = 1024

"""

with open(ROOT/"config"/"config.py","a") as f:
    f.write(config_update)

print("config updated")

config updated


In [99]:
prompt_code = '''
def build_prompt(question, contexts):

    context_text = "\\n\\n".join(
        [
            f"[Source {i+1}]\\n{chunk}"
            for i, chunk in enumerate(contexts)
        ]
    )

    prompt = f"""
You are an intelligent document assistant.

Answer ONLY using the information contained in the context below.

If the answer cannot be found in the context, reply:

"I could not find the answer in the provided document."

Context
-------
{context_text}

Question
--------
{question}

Instructions
------------
- Give a clear answer.
- Be concise.
- Do not hallucinate.
- Mention important details.
"""

    return prompt
'''

(ROOT/"core"/"prompt.py").write_text(prompt_code)

print("prompt.py created")

prompt.py created


In [100]:
gemini_code = '''
import os
import google.generativeai as genai

genai.configure(
    api_key=os.environ["GEMINI_API_KEY"]
)

class GeminiClient:

    def __init__(self):

        self.model = genai.GenerativeModel(
            "gemini-3.5-flash"
        )

    def generate(self, prompt):

        response = self.model.generate_content(

            prompt,

            generation_config={

                "temperature":0.2,

                "max_output_tokens":1024

            }

        )

        return response.text
'''

(ROOT/"core"/"gemini_client.py").write_text(gemini_code)

print("gemini_client.py created")

gemini_client.py created


In [101]:
rag_code = '''
from core.embedding import EmbeddingModel
from core.vector_store import VectorStore
from core.prompt import build_prompt
from core.gemini_client import GeminiClient

class RAGPipeline:

    def __init__(self):

        self.embedder = EmbeddingModel()

        self.store = VectorStore()

        self.store.load("vector_store")

        self.llm = GeminiClient()

    def ask(self, question, top_k=5):

        query = self.embedder.encode(question)

        results = self.store.search(query, k=top_k)

        contexts = [

            r["text"]

            for r in results

        ]

        prompt = build_prompt(

            question,

            contexts

        )

        answer = self.llm.generate(prompt)

        return {

            "answer": answer,

            "sources": results

        }
'''

(ROOT/"core"/"rag_pipeline.py").write_text(rag_code)

print("rag_pipeline.py created")

rag_pipeline.py created


In [102]:
app_code = '''
from core.rag_pipeline import RAGPipeline

rag = RAGPipeline()

print("="*70)

print("PDF CHATBOT")

print("="*70)

while True:

    question = input("\\nQuestion (exit to quit): ")

    if question.lower() == "exit":

        break

    response = rag.ask(question)

    print()

    print("="*70)

    print("ANSWER")

    print("="*70)

    print(response["answer"])

    print()

    print("="*70)

    print("SOURCES")

    print("="*70)

    for source in response["sources"]:

        print()

        print(f"File : {source['filename']}")

        print(f"Page : {source['page']}")

        print(f"Similarity : {source['score']:.3f}")

        print("-"*50)

        print(source["text"][:250])

        print()

'''

(ROOT/"app.py").write_text(app_code)

print("app.py created")

app.py created


In [103]:
!python app.py

2026-08-12 16:20:30.665821: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
PDF CHATBOT

Question (exit to quit): What is RAG?

ANSWER
Based on the provided documents, RAG is a cost-effective method and evolving research paradigm integrated within Large Language Models (LLMs) that surpasses the performance of native LLMs. 

Key details about RAG include:
* **Core Stages:** It analyzes key technologies in the core stages of "Retrieval," "Generation," and "Augmentation."
* **Evolutionary Stages:** It has evolved through three main stages: Naive RAG, Advanced RAG, and Modular RAG.
* **Development:** While initially limited to the inference stage, its enhancement has grown to incorporate LLM fine-tuning techniques.

SOURCES

File : RAG.pdf
Page : 1
S

## Part 6 — Professional Streamlit Web App

In [104]:
!pip -q install streamlit streamlit-option-menu watchdog

In [105]:
streamlit_code = r'''
import os
import tempfile
from pathlib import Path

import streamlit as st

from core.rag_pipeline import RAGPipeline

st.set_page_config(
    page_title="PDF Chatbot",
    page_icon="📄",
    layout="wide"
)

st.title("📄 Enterprise PDF Chatbot")

st.markdown(
    """
Ask questions about your uploaded documents using
Retrieval-Augmented Generation (RAG) powered by
Gemini 3.5 Flash.
"""
)

if "rag" not in st.session_state:
    st.session_state.rag = RAGPipeline()

if "history" not in st.session_state:
    st.session_state.history = []

uploaded_file = st.sidebar.file_uploader(
    "Upload PDF",
    type=["pdf"]
)

if uploaded_file:

    save_path = Path("data/raw") / uploaded_file.name

    with open(save_path,"wb") as f:
        f.write(uploaded_file.read())

    st.sidebar.success("PDF uploaded successfully.")

question = st.chat_input("Ask a question...")

if question:

    with st.spinner("Searching..."):

        response = st.session_state.rag.ask(question)

    st.session_state.history.append(
        (
            question,
            response
        )
    )

for question,response in st.session_state.history:

    with st.chat_message("user"):

        st.write(question)

    with st.chat_message("assistant"):

        st.write(response["answer"])

        with st.expander("Sources"):

            for source in response["sources"]:

                st.markdown(
                    f"""
**File:** {source['filename']}

**Page:** {source['page']}

**Similarity:** {source['score']:.3f}

---

{source['text']}
"""
                )
'''

(ROOT/"app.py").write_text(streamlit_code)

print("Streamlit app created.")

Streamlit app created.


In [106]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [107]:
import subprocess
import time

streamlit_process = subprocess.Popen(
    ["streamlit", "run", str(ROOT/"app.py"), "--server.port", "8501"]
)

time.sleep(5)

In [108]:
!./cloudflared tunnel --url http://localhost:8501

2026-08-12T16:21:21Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-12T16:21:21Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-12T16:21:26Z INF +--------------------------------------------------------------------------------------------+
2026-08-12T16:21:26Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-12T16:21:26Z INF |  https://oil-log-knight-vip.trycloudflare.com         

## Final Part 7 — Production Polish & GitHub Release

In [109]:
requirements = """
google-generativeai==0.8.5
langchain==0.3.27
langchain-community==0.3.29
langchain-text-splitters==0.3.11
sentence-transformers==5.1.0
faiss-cpu==1.12.0
pymupdf==1.26.4
python-dotenv==1.1.1
streamlit==1.49.1
tqdm==4.67.1
numpy>=1.26,<3
"""

(ROOT / "requirements.txt").write_text(requirements.strip() + "\n")

print("requirements.txt finalized")
print((ROOT / "requirements.txt").read_text())

requirements.txt finalized
google-generativeai==0.8.5
langchain==0.3.27
langchain-community==0.3.29
langchain-text-splitters==0.3.11
sentence-transformers==5.1.0
faiss-cpu==1.12.0
pymupdf==1.26.4
python-dotenv==1.1.1
streamlit==1.49.1
tqdm==4.67.1
numpy>=1.26,<3



In [110]:
dockerfile = """
FROM python:3.11-slim

# Prevent Python from writing .pyc files and buffering stdout
ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

WORKDIR /app

# System deps required by PyMuPDF / faiss-cpu wheels
RUN apt-get update && apt-get install -y --no-install-recommends \\
    build-essential \\
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

# Persisted at runtime via a mounted volume in production
RUN mkdir -p data/raw data/processed data/chunks vector_store logs

EXPOSE 8501

HEALTHCHECK CMD curl --fail http://localhost:8501/_stcore/health || exit 1

ENTRYPOINT ["streamlit", "run", "app.py", \\
    "--server.port=8501", \\
    "--server.address=0.0.0.0"]
"""

(ROOT / "Dockerfile").write_text(dockerfile.strip() + "\n")

print("Dockerfile created")

Dockerfile created


In [111]:
from datetime import datetime

license_text = f"""MIT License

Copyright (c) {datetime.now().year} Kunal Kirtak

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

(ROOT / "LICENSE").write_text(license_text)

print("LICENSE created")

LICENSE created


In [112]:
readme = """
# 📄 PDF Chatbot using Retrieval-Augmented Generation (RAG)

![Python](https://img.shields.io/badge/Python-3.11-blue)
![Gemini](https://img.shields.io/badge/LLM-Gemini%203.5%20Flash-orange)
![FAISS](https://img.shields.io/badge/VectorDB-FAISS-green)
![Streamlit](https://img.shields.io/badge/UI-Streamlit-red)
![License](https://img.shields.io/badge/License-MIT-lightgrey)

An enterprise-style RAG chatbot that lets you upload a PDF and ask natural
language questions about it. Answers are grounded in the document using
retrieval-augmented generation, with cited source chunks for every answer.

---

## Overview

The pipeline extracts text from a PDF, splits it into overlapping chunks,
embeds each chunk locally with `all-MiniLM-L6-v2`, indexes the vectors in a
FAISS store, and at query time retrieves the top-k most similar chunks to
ground a Gemini 3.5 Flash answer. Both a CLI demo and a full Streamlit app
are included.

## Architecture

```
PDF --> PyMuPDF extraction --> RecursiveCharacterTextSplitter (chunking)
     --> SentenceTransformer embeddings --> FAISS index (persisted to disk)

User question --> embed query --> FAISS similarity search --> top-k chunks
               --> prompt template --> Gemini 3.5 Flash --> answer + sources
```

## Folder Structure

```
01-PDF-Chatbot/
├── app.py                  # Streamlit web application
├── build_vector_store.py   # Builds and persists the FAISS index
├── chunk_documents.py       # Recursive chunking of ingested PDFs
├── ingest.py                # PDF text extraction
├── search_demo.py           # CLI similarity-search demo
├── requirements.txt
├── Dockerfile
├── LICENSE
├── .gitignore
├── config/
│   └── config.py            # Chunking, retrieval & model settings
├── core/
│   ├── logger.py
│   ├── utils.py
│   ├── loader.py
│   ├── chunker.py
│   ├── embedding.py         # all-MiniLM-L6-v2 wrapper
│   ├── vector_store.py      # FAISS build / save / load / search
│   ├── prompt.py            # RAG prompt template
│   ├── gemini_client.py     # Gemini 3.5 Flash wrapper
│   └── rag_pipeline.py      # End-to-end RAG orchestration
├── data/
│   ├── raw/                 # Uploaded PDFs
│   ├── processed/           # Extracted text (JSON)
│   └── chunks/               # Chunked text (JSON)
├── vector_store/             # faiss_index.bin + metadata.json
├── logs/
└── screenshots/
```

## Tech Stack

| Layer | Tool |
|---|---|
| LLM | Google Gemini 3.5 Flash (`google-generativeai`) |
| Embeddings | `sentence-transformers` (all-MiniLM-L6-v2) |
| Vector DB | FAISS (`faiss-cpu`) |
| PDF parsing | PyMuPDF |
| Chunking | LangChain `RecursiveCharacterTextSplitter` |
| UI | Streamlit |
| Infra | Docker |

## Installation

```bash
git clone https://github.com/<your-username>/01-PDF-Chatbot.git
cd 01-PDF-Chatbot

python -m venv venv
source venv/bin/activate      # Windows: venv\\Scripts\\activate

pip install -r requirements.txt

export GEMINI_API_KEY="your-api-key-here"   # Windows: set GEMINI_API_KEY=...
```

## Usage

**1. Ingest a PDF and build the index**

```bash
cp /path/to/your.pdf data/raw/
python ingest.py
python chunk_documents.py
python build_vector_store.py
```

**2. Ask questions from the CLI**

```bash
python search_demo.py
```

**3. Or launch the full web app**

```bash
streamlit run app.py
```

Then open the sidebar to upload a PDF and start chatting. Every answer
expands to show the retrieved source chunks with similarity scores.

## Run with Docker

```bash
docker build -t pdf-chatbot .
docker run -p 8501:8501 -e GEMINI_API_KEY=your-api-key-here pdf-chatbot
```

## Example Output

```
Question: What is the termination clause in this contract?

Answer: Either party may terminate the agreement with 30 days' written
notice, or immediately in the event of a material breach that remains
uncured for 15 days after notice.

Sources:
[1] contract.pdf — page 4 — score 0.812
[2] contract.pdf — page 5 — score 0.774
```

## Screenshots

> _Add screenshots of the Streamlit UI here, e.g._
> `![Chat UI](screenshots/chat_ui.png)`

## Future Improvements

- Swap FAISS `IndexFlatIP` for an approximate index (HNSW/IVF) for large corpora
- Support multi-PDF collections with per-document filtering
- Stream Gemini responses token-by-token in the UI
- Add automated evaluation (retrieval precision/recall, answer faithfulness)
- Re-rank retrieved chunks with a cross-encoder before prompting
- Persist chat history per user session (SQLite/Postgres)

## License

Released under the [MIT License](LICENSE).
"""

(ROOT / "README.md").write_text(readme.strip() + "\\n")

print("README.md created")

README.md created


In [113]:
required_files = [
    "app.py",
    "ingest.py",
    "chunk_documents.py",
    "build_vector_store.py",
    "search_demo.py",
    "requirements.txt",
    "README.md",
    "Dockerfile",
    "LICENSE",
    ".gitignore",
    "config/config.py",
    "core/logger.py",
    "core/utils.py",
    "core/pdf_loader.py",
    "core/chunker.py",
    "core/embedding.py",
    "core/vector_store.py",
    "core/prompt.py",
    "core/gemini_client.py",
    "core/rag_pipeline.py",
]

print("Repository check")
print("=" * 60)

missing = []

for rel_path in required_files:
    full_path = ROOT / rel_path
    exists = full_path.exists()
    status = "OK" if exists else "MISSING"
    print(f"[{status}] {rel_path}")
    if not exists:
        missing.append(rel_path)

print("=" * 60)

if missing:
    print(f"{len(missing)} file(s) missing. Re-run the earlier cells that generate them.")
else:
    print("All required project files are present.")

Repository check
[OK] app.py
[OK] ingest.py
[OK] chunk_documents.py
[OK] build_vector_store.py
[OK] search_demo.py
[OK] requirements.txt
[OK] README.md
[OK] Dockerfile
[OK] LICENSE
[OK] .gitignore
[OK] config/config.py
[OK] core/logger.py
[OK] core/utils.py
[OK] core/pdf_loader.py
[OK] core/chunker.py
[OK] core/embedding.py
[OK] core/vector_store.py
[OK] core/prompt.py
[OK] core/gemini_client.py
[OK] core/rag_pipeline.py
All required project files are present.


In [114]:
import shutil

archive_path = shutil.make_archive(
    base_name=str(ROOT.parent / ROOT.name),
    format="zip",
    root_dir=str(ROOT),
)

print(f"Repository packaged: {archive_path}")

try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print("Not running in Colab — retrieve the zip from the path above.")

Repository packaged: /content/drive/MyDrive/01-PDF-Chatbot.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>